In [2]:
import os
import pandas as pd
from tqdm import tqdm
from huggingface_hub import hf_hub_download

# --- 配置区 ---
REPO_ID = "DeepFense/SpeechFake"
SAVE_DIR = "./Speechfake"
# 挑选你认为最先进/主流的模型
MAINSTREAM_MODELS = [
    'CosyVoice', 'ChatTTS', 'GPT-SoVITS', 'StyleTTS2', 
    'BigVGAN', 'HiFi-GAN', 'TorToiSe', 'OpenVoice'
]

os.makedirs(SAVE_DIR, exist_ok=True)

# 1. 下载并读取 Baseline 的 English 测试集索引 (Parquet 格式)
print("正在获取 English 数据索引...")
metadata_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="baseline/test_en.parquet", # 也可以换成 train_en.parquet
    repo_type="dataset"
)
df = pd.read_parquet(metadata_path)

# 2. 筛选逻辑
# 真实音频 (bonafide)
real_df = df[df['label'] == 'bonafide']
# 合成音频 (spoof) - 仅限主流模型
# 注意：如果 df 里没有 model 列，请先 print(df.columns) 检查。
# 通常 SpeechFake 的 path 里就包含了模型名称，我们可以从 path 提取
if 'model' not in df.columns:
    df['model_name'] = df['path'].apply(lambda x: x.split('/')[1] if 'Real' not in x else 'real')
    fake_df = df[(df['label'] == 'spoof') & (df['model_name'].isin(MAINSTREAM_MODELS))]
else:
    fake_df = df[(df['label'] == 'spoof') & (df['model'].isin(MAINSTREAM_MODELS))]

# 3. 设定 1:5 比例抽样 (目标总量约 20GB)
# 经验法则：16kHz wav 每秒约 32KB。平均 5s 一个文件约 160KB。
# 20GB 约需 125,000 个文件。
# 取 20,000 个 Real : 100,000 个 Fake
num_real = min(len(real_df), 20000)
num_fake = num_real * 5

selected_real = real_df.sample(n=num_real, random_state=42)
selected_fake = fake_df.sample(n=min(len(fake_df), num_fake), random_state=42)
download_list = pd.concat([selected_real, selected_fake])

print(f"筛选完成！计划下载 Real: {len(selected_real)}, Fake: {len(selected_fake)}")

# 4. 执行精准下载
print("开始下载音频文件...")
for _, row in tqdm(download_list.iterrows(), total=len(download_list)):
    file_path = row['path']
    try:
        # 下载到本地并保持目录结构
        hf_hub_download(
            repo_id=REPO_ID,
            filename=file_path,
            repo_type="dataset",
            local_dir=SAVE_DIR,
            local_dir_use_symlinks=False # 设为 False 直接复制文件到本地
        )
    except Exception as e:
        # 跳过缺失或报错的文件
        continue

print(f"下载任务结束。数据保存在: {os.path.abspath(SAVE_DIR)}")

正在获取 English 数据索引...
筛选完成！计划下载 Real: 19200, Fake: 40253
开始下载音频文件...


  0%|          | 0/59453 [00:00<?, ?it/s]/home/jesse/Projects/MyEnvs/finquest_2026/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
100%|██████████| 59453/59453 [5:37:24<00:00,  2.94it/s]  

下载任务结束。数据保存在: /home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/Speechfake
